In [0]:
# Notebook: 03_generate_inventory
# Purpose: Simulate daily inventory snapshots per dark store for ZapFlow platform
# Layer: Simulation

import random
from datetime import datetime, timedelta
from pyspark.sql.types import *

# ── Configuration ─────────────────────────────────────────────
OUTPUT_PATH = "/Volumes/zapflow_raw/raw_data/raw_files/inventory/"
SEED = 42
random.seed(SEED)

# ── Product categories quick commerce typically carries ────────
CATEGORIES = {
    "Fruits & Vegetables": ["Tomato", "Onion", "Potato", "Banana", "Apple"],
    "Dairy":               ["Milk", "Curd", "Butter", "Paneer", "Cheese"],
    "Snacks":              ["Chips", "Biscuits", "Namkeen", "Chocolates", "Cookies"],
    "Beverages":           ["Water", "Juice", "Soda", "Tea", "Coffee"],
    "Personal Care":       ["Shampoo", "Soap", "Toothpaste", "Facewash", "Sanitizer"]
}

# Store IDs matching what we generated
STORE_IDS = [f"STORE_{i:04d}" for i in range(1, 51)]

# Simulate 90 days of inventory snapshots
START_DATE = datetime(2024, 1, 1)
NUM_DAYS   = 90

# ── Helper ─────────────────────────────────────────────────────
def introduce_data_quality_issues(record, counter):
    issue_type = None
    if counter % 40 == 0:
        record["units_available"] = -1
        issue_type = "negative_stock"
    elif counter % 60 == 0:
        record["reorder_level"] = None
        issue_type = "missing_reorder_level"
    record["_data_quality_issue"] = issue_type
    return record

# ── Generate records ───────────────────────────────────────────
def generate_inventory():
    records  = []
    counter  = 1
    inv_id   = 1

    for day_offset in range(NUM_DAYS):
        snapshot_date = (START_DATE + timedelta(days=day_offset)).date()

        for store_id in STORE_IDS:
            for category, products in CATEGORIES.items():
                for product in products:

                    # Weekends have lower stock replenishment
                    is_weekend     = snapshot_date.weekday() >= 5
                    max_stock      = 100 if not is_weekend else 70

                    units_available = random.randint(0, max_stock)
                    reorder_level   = random.randint(10, 30)
                    units_sold      = random.randint(0, max_stock - units_available)

                    record = {
                        "inventory_id":     f"INV_{inv_id:08d}",
                        "store_id":         store_id,
                        "product_name":     product,
                        "category":         category,
                        "snapshot_date":    str(snapshot_date),
                        "units_available":  units_available,
                        "units_sold":       units_sold,
                        "reorder_level":    reorder_level,
                        "is_out_of_stock":  units_available == 0,
                        "is_weekend":       is_weekend,
                        "_data_quality_issue": None
                    }

                    record = introduce_data_quality_issues(record, counter)
                    records.append(record)
                    inv_id  += 1
                    counter += 1

    return records

# ── Main execution ─────────────────────────────────────────────
print("Generating inventory data...")
print("This will take a moment — 90 days × 50 stores × 25 products...")

inventory_records = generate_inventory()

schema = StructType([
    StructField("inventory_id",        StringType(),  False),
    StructField("store_id",            StringType(),  True),
    StructField("product_name",        StringType(),  True),
    StructField("category",            StringType(),  True),
    StructField("snapshot_date",       StringType(),  True),
    StructField("units_available",     IntegerType(), True),
    StructField("units_sold",          IntegerType(), True),
    StructField("reorder_level",       IntegerType(), True),
    StructField("is_out_of_stock",     BooleanType(), True),
    StructField("is_weekend",          BooleanType(), True),
    StructField("_data_quality_issue", StringType(),  True),
])

df_inventory = spark.createDataFrame(inventory_records, schema=schema)

# ── Save to Volume ─────────────────────────────────────────────
(df_inventory
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"✅ Generated {len(inventory_records)} inventory records")
print(f"✅ Saved to {OUTPUT_PATH}")

# ── Sanity checks ──────────────────────────────────────────────
print("\n── Records per category ──")
df_inventory.groupBy("category").count().orderBy("count", ascending=False).show()

print("── Out of stock summary ──")
df_inventory.groupBy("is_out_of_stock").count().show()

print("── Data quality issues ──")
df_inventory.groupBy("_data_quality_issue").count().show()

print("── Sample records ──")
df_inventory.show(5, truncate=False)